<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>Sistema Autocompletado con N-Gramas (NLTK)</h2>
</div>


En este notebook construiremos un sistema de **autocompletado** usando los modelos de lenguaje de **n-gramas** que ofrece la librería [NLTK](https://www.nltk.org/api/nltk.lm.html).

A diferencia de implementar los n-gramas desde cero, aquí aprovecharemos el módulo `nltk.lm` (Language Model), que incluye:
- Generación de n-gramas y padding de oraciones.
- Construcción del vocabulario.
- Modelos con suavizado (MLE, Laplace, Kneser-Ney, etc.).
- Predicción de la siguiente palabra y generación de texto.

**Objetivos:**
1. Preparar y tokenizar un corpus de texto.
2. Entrenar un modelo de lenguaje de n-gramas con NLTK.
3. Calcular probabilidades y perplejidad.
4. Construir una función de autocompletado que sugiera la siguiente palabra.

<a name='1'></a>
## 1. Importación de librerías

Descargaremos los recursos necesarios (`punkt` para la tokenización).

In [ ]:
# Descomenta la siguiente línea si necesitas instalar NLTK
# %pip install nltk

import nltk

# Descarga los recursos necesarios para la tokenizacion
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Importa las herramientas del modulo de modelos de lenguaje de NLTK
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

<a name='2'></a>
## 2. Preparación del corpus

Usaremos un pequeño corpus de ejemplo en español. En un caso real, cargarías un archivo de texto grande (por ejemplo, tweets, noticias o libros).

El flujo de preprocesamiento es:
1. Dividir el texto en oraciones.
2. Tokenizar cada oración en palabras en minúsculas (Aplicar el preprocesamiento visto).

In [ ]:
# Corpus de ejemplo: cada cadena es una oracion
corpus = [
    "El procesamiento del lenguaje natural es una rama de la inteligencia artificial.",
    "Los modelos de lenguaje predicen la siguiente palabra en una secuencia.",
    "El autocompletado utiliza modelos de lenguaje para sugerir palabras.",
    "Los n-gramas son una tecnica simple pero poderosa para modelar el lenguaje.",
    "La inteligencia artificial transforma la manera en que interactuamos con la tecnologia.",
    "Un modelo de lenguaje asigna una probabilidad a una secuencia de palabras.",
    "El aprendizaje automatico es fundamental en el procesamiento del lenguaje natural.",
    "Las redes neuronales han mejorado mucho los modelos de lenguaje modernos."
]

# Tokeniza cada oracion en palabras en minusculas
tokenized_text = [word_tokenize(sentence.lower(), language='spanish') for sentence in corpus]

print("Numero de oraciones:", len(tokenized_text))
print("Primera oracion tokenizada:")
print(tokenized_text[0])

<a name='3'></a>
## 3. Construcción del modelo de n-gramas con NLTK

NLTK ofrece la función `padded_everygram_pipeline`, que a partir de las oraciones tokenizadas genera dos cosas:
- Los **n-gramas** (con padding de inicio `<s>` y fin `</s>`) para entrenar.
- El **vocabulario** (un generador de todas las palabras).

Definimos el orden `n` del modelo (por ejemplo, `n=2` para bigramas, `n=3` para trigramas).

In [ ]:
# Orden del modelo de n-gramas (3 = trigramas)
n = 3

# Genera los n-gramas con padding y el vocabulario a partir del texto tokenizado


<a name='4'></a>
## 4. Entrenamiento del modelo

Crearemos un modelo con **suavizado de Laplace** (add-one). Esto evita probabilidades de cero para n-gramas no vistos.

Otras opciones disponibles en NLTK son:
- `MLE`: Maxima Verosimilitud (sin suavizado).
- `Laplace`: suavizado add-one.
- `KneserNeyInterpolated`: suavizado Kneser-Ney interpolado.

In [ ]:
# Crea el modelo de lenguaje con suavizado de Laplace


# Entrena el modelo con los n-gramas y el vocabulario


<a name='5'></a>
## 5. Cálculo de probabilidades

Una vez entrenado el modelo, podemos consultar:
- `modelo.score(palabra, contexto)`: la probabilidad de una palabra dado un contexto.
- `modelo.logscore(palabra, contexto)`: el logaritmo de esa probabilidad.

In [ ]:
# Probabilidad de la palabra 'lenguaje' dado el contexto ['modelos', 'de']


# Probabilidad de la palabra 'de' dado el contexto ['modelos']

# Log-probabilidad


<a name='6'></a>
## 6. Perplejidad

La **perplejidad** mide qué tan bien predice el modelo una secuencia de prueba. Cuanto menor sea la perplejidad, mejor es el modelo.

Para evaluar, primero convertimos una oración de prueba en n-gramas.

In [ ]:
# Oracion de prueba tokenizada

# Convierte la oracion en n-gramas del mismo orden que el modelo


# Calcula la perplejidad


<a name='7'></a>
## 7. Generación de texto

El método `modelo.generate()` permite generar texto de manera automática a partir del modelo entrenado. Podemos darle un contexto inicial (`text_seed`).

In [ ]:
import random

# Fija la semilla para reproducibilidad
random.seed(42)

# Genera 10 palabras a partir del contexto inicial
palabras_generadas = modelo.generate(10, text_seed=['los', 'modelos'], random_seed=42)
print("Texto generado:")
print('los modelos ' + ' '.join(palabras_generadas))

<a name='8'></a>
## 8. Sistema de autocompletado

Ahora construiremos una función de autocompletado. Dado un texto parcial, la función:
1. Toma las últimas `n-1` palabras como contexto.
2. Recorre todo el vocabulario calculando la probabilidad de cada palabra.
3. Devuelve las palabras con mayor probabilidad.

También permitimos filtrar por las primeras letras (`start_with`), igual que un teclado predictivo.

In [ ]:
def autocompletar(texto_parcial, modelo, top_k=3, start_with=None):
    # Tokeniza el texto parcial en minusculas
    tokens = word_tokenize(texto_parcial.lower(), language='spanish')

    # Toma las ultimas n-1 palabras como contexto
    contexto = tokens[-(modelo.order - 1):]

    # Calcula la probabilidad de cada palabra del vocabulario
    sugerencias = []
    for palabra in modelo.vocab:
        # Ignora los tokens especiales de inicio, fin y desconocido
        if palabra in ('<s>', '</s>', '<UNK>'):
            continue
        # Aplica el filtro por letras iniciales si se especifica
        if start_with is not None and not palabra.startswith(start_with):
            continue
        prob = modelo.score(palabra, contexto)
        if prob > 0:
            sugerencias.append((palabra, prob))

    # Ordena de mayor a menor probabilidad y devuelve las top_k
    sugerencias.sort(key=lambda x: x[1], reverse=True)
    return sugerencias[:top_k]

In [ ]:
# Prueba el autocompletado


In [ ]:
# Prueba el autocompletado filtrando por letra inicial
texto = "el procesamiento del"
